# 1. Data and labels

This notebook reproduces the reported figures of the sections *Dataset used for evaluation: arm one*
and *Measurement and labelling summary*. It reads the shipped exports only and fits no model.
Each quoted figure goes through `check()`, so a mismatch stops the notebook.

In [1]:
from common import *
from sklearn.metrics import cohen_kappa_score

PAIRS = load_pairs()
BASE = PAIRS[PAIRS.contrast_type == "base_mut"]
MUT = PAIRS[PAIRS.contrast_type == "mut_mut"]
MEDIANS = pd.read_csv(DATA / "redmi9t_role_medians.csv", dtype={"group": str})
REVIEW = load(RES / "posthoc_sensitivity_checks.json")

## Corpus

A *contrast* is a pair of variants of one widget. `base_mut` compares the developer's original
widget with one of its controlled variants and is the primary stratum. `mut_mut` compares two variants.

In [2]:
check("contrasts", len(PAIRS), 1454)
check("seed groups", PAIRS.group.nunique(), 43)
check("repositories", PAIRS.project.nunique(), 8)
check("base_mut contrasts", len(BASE), 332)
check("mut_mut contrasts", len(MUT), 1122)
check("roles analysed", len(MEDIANS), 375)
check("executions per role (all roles)", (MEDIANS.n == 15).mean(), 1.0)
check("executions in total", int(MEDIANS.n.sum()), 5625)

share = BASE.project.value_counts(normalize=True)
check("largest repository, base_mut contrasts", BASE.project.value_counts().max(), 73)
check("largest repository share", share.max(), 0.220, 5e-4)
check("effective number of repositories", 1 / (share ** 2).sum(), 6.16, 5e-3)
rate = BASE.groupby("project").label.mean()
check("per-repository slower share, min", rate.min(), 0.250)
check("per-repository slower share, max", rate.max(), 0.500)
BASE.groupby("project").agg(contrasts=("label", "size"), slower=("label", "mean")).round(3).sort_values("contrasts", ascending=False)

,contrasts,slower
project,,
SatoshiPortal_bullbitcoin-mobile,73,0.452
deckerst_aves,71,0.366
DompetApp_Dompet.flutter,56,0.446
fluttercandies_wechat_flutter,47,0.362
Hash-Studios_Prism,31,0.484
WheretoSleepinNJU_NJU-Class-Shedule-Flutter,30,0.500
nknorg_nMobile,16,0.375
amugofjava_anytime_podcast_player,8,0.250


## The inputs are the exported files

`data/redmi9t_snapshot_manifest.json` was written when the measurements were exported. It records the sha256 of the
two arm-1 exports and the role counts. The medians file is listed there under its earlier name, `redmi9t_medians_ds.csv`.

In [3]:
import hashlib

SNAP = load(DATA / "redmi9t_snapshot_manifest.json")
renamed = {"redmi9t_medians_ds.csv": "redmi9t_role_medians.csv"}
for name, rec in SNAP["exports"].items():
    digest = hashlib.sha256((DATA / renamed.get(name, name)).read_bytes()).hexdigest()
    check(f"{renamed.get(name, name)} matches its exported sha256", digest == rec["sha256"], True)
grid = SNAP["run_grid"]
check("nominal roles", grid["nominal_roles"], 387)
check("generated roles", grid["generated_roles"], 377)
check("analysed roles, recorded = recomputed", grid["analysed_roles"], len(MEDIANS))
check("executions reconciled with the raw logs, mismatches", SNAP["reconciliation"]["mismatches"], 0)

## The label, recomputed from the medians

Each role was executed 15 times, giving 15 execution medians. The Cliff's δ of the variant over the
original is the share of the 15 × 15 pairs in which the variant is slower minus the share in which it is faster.
Label 1 (*slower*) when δ exceeds 0.33, the small/medium boundary of the Romano scale. Every one of the 1,454
contrasts is recomputed here from the stored medians and compared with the exported value.

In [4]:
med_by_role = {r.sample_id: np.array(json.loads(r.medians)) for r in MEDIANS.itertuples()}

delta = np.array([cliffs_delta(med_by_role[f"{r.group}/{r.right}"], med_by_role[f"{r.group}/{r.left}"])
                  for r in PAIRS.itertuples()])
check("largest difference to the exported delta, all 1,454 contrasts", np.abs(delta - PAIRS.cliffs_delta).max(), 0.0, 1e-9)
check("labels that differ from the rule delta > 0.33", ((delta > CUT) != PAIRS.label.astype(bool)).sum(), 0)

The ambiguity band is |δ − 0.33| < 0.10. Prevalence is the share of *slower* labels.

In [5]:
check("base_mut slower", int(BASE.label.sum()), 139)
check("base_mut prevalence", BASE.label.mean(), 0.419, 5e-4)
check("mut_mut prevalence", MUT.label.mean(), 0.495, 5e-4)
ambiguous = (BASE.cliffs_delta - CUT).abs() < 0.10
check("ambiguous contrasts (within 0.10 of 0.33)", ambiguous.sum(), 42)
check("ambiguous share", ambiguous.mean(), 0.127, 5e-4)

width = BASE.delta_ci_high - BASE.delta_ci_low
check("median width of a contrast's 95% bootstrap interval", width.median(), 0.769, 5e-4)
check("intervals containing 0.33", ((BASE.delta_ci_low <= CUT) & (BASE.delta_ci_high >= CUT)).sum(), 170)
print(BASE.effect_magnitude.value_counts().to_string())

effect_magnitude
large         148
negligible     77
small          77
medium         30


## Sensitivity of the labels to the rule

Cohen's κ of each alternative rule against the primary one, over all 1,454 contrasts; the
*slower* count and prevalence are for `base_mut`.

In [6]:
rules = {
    "primary: delta > 0.33": PAIRS.cliffs_delta > CUT,
    "plus q < 0.05": (PAIRS.cliffs_delta > CUT) & (PAIRS.q_bh < 0.05),
    "strict: delta > 0.474": PAIRS.cliffs_delta > 0.474,
    "interval clears 0.33": PAIRS.delta_ci_low > CUT,
}
rows = []
for name, r in rules.items():
    r = r.astype(int)
    b = r[PAIRS.contrast_type == "base_mut"]
    rows.append({"rule": name, "kappa": cohen_kappa_score(PAIRS.label, r), "slower": int(b.sum()), "prevalence": b.mean()})
sens = pd.DataFrame(rows).set_index("rule")
check("kappa, plus q < 0.05", sens.kappa.iloc[1], 0.932, 5e-4)
check("labels changed by the q rule, all strata", int((rules["primary: delta > 0.33"] != rules["plus q < 0.05"]).sum()), 49)
check("kappa, delta > 0.474", sens.kappa.iloc[2], 0.909, 5e-4)
check("kappa, interval clears 0.33", sens.kappa.iloc[3], 0.765, 5e-4)
check("prevalence, interval clears 0.33", sens.prevalence.iloc[3], 0.250, 5e-4)
check("smallest delta surviving q < 0.05", PAIRS[rules["plus q < 0.05"]].cliffs_delta.min(), 0.436, 5e-4)
sens.round(3)

,kappa,slower,prevalence
rule,,,
primary: delta > 0.33,1.000,139,0.419
plus q < 0.05,0.932,128,0.386
strict: delta > 0.474,0.909,119,0.358
interval clears 0.33,0.765,83,0.250


## Drift with schedule position

Each session's log median is regressed on schedule position and the trend removed, from the per-execution medians
and positions in `data/redmi9t_executions.csv`. The models refitted on the adjusted labels are a frozen post-hoc
result (`results/posthoc_sensitivity_checks.json`).

In [7]:
EX1 = load_executions("redmi9t_executions.csv")
_, ADJ1 = drift_adjusted(EX1)
ok = EX1[EX1.status == "ok"].sort_values(["group", "role", "exec_index"])
RAW1 = {k: s.median_us.to_numpy() for k, s in ok.groupby(["group", "role"])}
raw = np.array([cliffs_delta(RAW1[(r.group, r.right)], RAW1[(r.group, r.left)]) for r in BASE.itertuples()])
adj = np.array([cliffs_delta(ADJ1[(r.group, r.right)], ADJ1[(r.group, r.left)]) for r in BASE.itertuples()])
check("per-execution medians reproduce the exported delta", np.abs(raw - BASE.cliffs_delta).max(), 0.0, 1e-12)
check("labels changed by drift adjustment", int(((raw > CUT) != (adj > CUT)).sum()), 10)
check("of 332", len(BASE), 332)
check("pairs whose delta changes sign", int((np.sign(raw) * np.sign(adj) < 0).sum()), 21)
check("sign changes clear of 0.33 on both readings",
      int(((np.sign(raw) * np.sign(adj) < 0) & (np.abs(raw) > CUT) & (np.abs(adj) > CUT)).sum()), 0)
check("prevalence, raw", (raw > CUT).mean(), 0.419, 5e-4)
check("prevalence, adjusted", (adj > CUT).mean(), 0.431, 5e-4)
d = REVIEW["B_position_adjusted_labels_arm1"]
check("labels changed (frozen refit input)", d["labels_changed"], 10)
pd.DataFrame(d["rerun_on_adjusted_labels"]).T.round(3)

,mcc,roc_auc
lightgbm,0.394,0.781
logreg_balanced,0.432,0.758
random_forest,0.452,0.802
rbf_svm,0.405,0.780


## Measurement quality

Role medians and their spread are recomputed from the 375 stored medians vectors; the warm-up and
trend checks are frozen post-hoc results over every execution. The thermal gate and battery readings come from the
header the runner writes as each execution starts (`data/execution_start_readings.csv`): the hottest thermal zone,
read just after the gate releases the execution, and the battery level. The extractor parity under `spm` 0.7.1 is
the result of `scripts/extractor_parity.py`.

In [8]:
med = MEDIANS.medians.map(lambda s: np.array(json.loads(s)))
role_median = med.map(np.median)
iqr_ratio = med.map(lambda v: (np.percentile(v, 75) - np.percentile(v, 25)) / np.median(v))
check("smallest role median (us)", role_median.min(), 299.0)
check("largest role median (us)", role_median.max(), 9095.5)
check("median IQR / median", iqr_ratio.median(), 0.0229, 5e-5)

w = REVIEW["D_warmup_arm1"]
check("zero-span timeouts", w["zero_span_timeouts"], 0)
check("executions analysed", w["analysed_executions"], 5625)
check("median ratio, rebuilds 1-10 over kept 20", w["ratio_median_first10_over_kept20"]["50"], 0.998, 5e-4)
check("5th percentile of that ratio", w["ratio_median_first10_over_kept20"]["5"], 0.870, 5e-4)
check("95th percentile of that ratio", w["ratio_median_first10_over_kept20"]["95"], 1.089, 5e-4)
check("executions with a rank trend at 5%", w["spearman_trend_within_kept_20"]["share_p_below_0.05"], 0.060, 5e-4)

START = pd.read_csv(DATA / "execution_start_readings.csv", dtype={"group": str})
by_arm = START.groupby("arm")
check("executions with a start reading, arm one", by_arm.size()["arm1"], 5655)
check("hottest zone at execution start, arm one, max (C)", by_arm.soc_temp_c.max()["arm1"], 63.0)
check("hottest zone at execution start, arm two, max (C)", by_arm.soc_temp_c.max()["arm2"], 51.3)
check("battery at execution start, min (%)", START.battery_pct.min(), 30)
check("battery at execution start, max (%)", START.battery_pct.max(), 85)

PARITY = load(RES / "extractor_parity.json")
check("arm-one roles re-extracted under spm 0.7.1", PARITY["n_roles_compared"], 377)
check("roles with any field different", PARITY["n_mismatching_roles"], 0)

## Figures

The figures for this notebook's sections, written to `figures/` as PNG (300 dpi) and SVG. They plot only values checked above. The images carry no title or note; the caption in the report explains them.

In [9]:
plt = figure_style()
SHORT = lambda p: p.split("_", 1)[1]
NAMES = dict(zip([f"mutation_{i}" for i in range(1, 9)], ["nest_deeper", "all_const", "drop_const", "eager_list",
             "extract_widgets", "inline_helpers", "add_siblings", "layout_builder"]))

# Base-to-variant contrasts per application, and the slower share per directive.
comp = BASE.groupby("project").label.agg(["size", "sum"]).sort_values("size")
rate = BASE.groupby("right").label.agg(["size", "mean"]).sort_values("mean")
check("figure: contrasts plotted", comp["size"].sum(), 332)
check("figure: lowest directive share", rate["mean"].min(), 0.051, 5e-4)
check("figure: highest directive share", rate["mean"].max(), 0.930, 5e-4)
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1.15, 1]})
pos = np.arange(len(comp))
a.barh(pos, comp["sum"], color=DARK, label="Slower")
a.barh(pos, comp["size"] - comp["sum"], left=comp["sum"], color=LIGHT, edgecolor=MID, hatch="///",
       label="Faster or stable")
a.set_yticks(pos, [SHORT(p) for p in comp.index], fontsize=7.5)
a.set_xlabel("base-to-variant contrasts")
a.set_title("(a)", loc="left")
a.legend(frameon=False, fontsize=7.5, loc="lower right")
pos = np.arange(len(rate))
b.barh(pos, rate["mean"], color=MID)
b.set_yticks(pos, [f"{NAMES[d]} (n={n})" for d, n in rate["size"].items()], fontsize=7.5)
b.axvline(BASE.label.mean(), color=DARK, ls="--", lw=0.9)
b.set_xlim(0, 1)
b.set_xlabel("share of contrasts labelled Slower")
b.set_title("(b)", loc="left")
fig.tight_layout()
save_figure(fig, "corpus_composition")

# Cliff's delta over every arm-one contrast, with the 0.33 threshold.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
bins = np.linspace(-1, 1, 41)
for ax, part, letter in ((axes[0], BASE, "(a)"), (axes[1], MUT, "(b)")):
    ax.axvspan(-CUT, CUT, color=LIGHT, alpha=0.6, lw=0)
    ax.hist(part.cliffs_delta, bins=bins, color=MID, edgecolor="white")
    ax.axvline(CUT, color=DARK, ls="--", lw=0.9)
    ax.set_xlim(-1, 1)
    ax.set_xlabel("Cliff's δ (right side against left side)")
    ax.set_title(letter, loc="left")
axes[0].set_ylabel("contrasts")
fig.tight_layout()
save_figure(fig, "cliffs_delta")


wrote figures/corpus_composition.png and .svg


wrote figures/cliffs_delta.png and .svg


In [10]:
report()

61 / 61 reported figures reproduced


,claim,value,reported,ok
0,contrasts,1454,1454,True
1,seed groups,43,43,True
2,repositories,8,8,True
3,base_mut contrasts,332,332,True
4,mut_mut contrasts,1122,1122,True
...,...,...,...,...
56,arm-one roles re-extracted under spm 0.7.1,377,377,True
57,roles with any field different,0,0,True
58,figure: contrasts plotted,332,332,True
59,figure: lowest directive share,0.051282,0.051,True
